In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    .appName('pyspark-pylot')
    .master('local[*]')
    .getOrCreate()
)

spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/13 19:57:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
emp_data = [
    ["001","101","John Doe","30","Male","50000","2015-01-01"],
    ["002","101","Jane Smith","25","Female","45000","2016-02-15"],
    ["003","102","Bob Brown","35","Male","55000","2014-05-01"],
    ["004","102","Alice Lee","28","Female","48000","2017-09-30"],
    ["005","103","Jack Chan","40","Male","60000","2013-04-01"],
    ["006","103","Jill Wong","32","Female","52000","2018-07-01"],
    ["007","101","James Johnson","42","Male","70000","2012-03-15"],
    ["008","102","Kate Kim","29","Female","51000","2019-10-01"],
    ["009","103","Tom Tan","33","Male","58000","2016-06-01"],
    ["010","104","Lisa Lee","27","Female","47000","2018-08-01"],
    ["011","104","David Park","38","Male","65000","2015-11-01"],
    ["012","105","Susan Chen","31","Female","54000","2017-02-15"],
    ["013","106","Brian Kim","45","Male","75000","2011-07-01"],
    ["014","107","Emily Lee","26","Female","46000","2019-01-01"],
    ["015","106","Michael Lee","37","Male","63000","2014-09-30"],
    ["016","107","Kelly Zhang","30","Female","49000","2018-04-01"],
    ["017","105","George Wang","34","Male","57000","2016-03-15"],
    ["018","104","Nancy Liu","29","Female","50000","2017-06-01"],
    ["019","103","Steven Chen","36","Male","62000","2015-08-01"],
    ["020","102","Grace Kim","32","Female","53000","2018-11-01"]
]

emp_schema = "employee_id string, department_id string, name string, age string, gender string, salary string, hire_date string"

emp_df = spark.createDataFrame(data=emp_data, schema=emp_schema)

In [3]:
# Dataframe schema

emp_df.printSchema()

root
 |-- employee_id: string (nullable = true)
 |-- department_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- age: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: string (nullable = true)
 |-- hire_date: string (nullable = true)



In [4]:
# Dataframe schema on native spark structure

emp_df.schema

StructType([StructField('employee_id', StringType(), True), StructField('department_id', StringType(), True), StructField('name', StringType(), True), StructField('age', StringType(), True), StructField('gender', StringType(), True), StructField('salary', StringType(), True), StructField('hire_date', StringType(), True)])

In [5]:
# String schema and spark structure schema definitios
string_schema = 'name string, age int'

from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, DoubleType

spark_schema = StructType([
    StructField('name', StringType(), True),
    StructField('age', IntegerType(), True)
])

from pyspark.sql.types import _parse_datatype_string

spark_schema = _parse_datatype_string(string_schema)
print(spark_schema)

StructType([StructField('name', StringType(), True), StructField('age', IntegerType(), True)])


In [6]:
# Spark columns and expressions

from pyspark.sql.functions import col, expr

print(col('salary'), expr('salary'), emp_df.salary, emp_df['salary'])

Column<'salary'> Column<'salary'> Column<'salary'> Column<'salary'>


In [7]:
# Selecting columns

emp_df.select(col('employee_id'), expr('name'), emp_df.age, emp_df['salary']).show()

+-----------+-------------+---+------+
|employee_id|         name|age|salary|
+-----------+-------------+---+------+
|        001|     John Doe| 30| 50000|
|        002|   Jane Smith| 25| 45000|
|        003|    Bob Brown| 35| 55000|
|        004|    Alice Lee| 28| 48000|
|        005|    Jack Chan| 40| 60000|
|        006|    Jill Wong| 32| 52000|
|        007|James Johnson| 42| 70000|
|        008|     Kate Kim| 29| 51000|
|        009|      Tom Tan| 33| 58000|
|        010|     Lisa Lee| 27| 47000|
|        011|   David Park| 38| 65000|
|        012|   Susan Chen| 31| 54000|
|        013|    Brian Kim| 45| 75000|
|        014|    Emily Lee| 26| 46000|
|        015|  Michael Lee| 37| 63000|
|        016|  Kelly Zhang| 30| 49000|
|        017|  George Wang| 34| 57000|
|        018|    Nancy Liu| 29| 50000|
|        019|  Steven Chen| 36| 62000|
|        020|    Grace Kim| 32| 53000|
+-----------+-------------+---+------+



In [8]:
# Using expressions

emp_simple_df = emp_df.select(expr('employee_id as emp_id'), col('name'), expr('cast(age as int) as age'), col('salary'))
emp_simple_df.printSchema()

root
 |-- emp_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- salary: string (nullable = true)



In [9]:
# Using selectExpr

emp_simple_df_1 = emp_df.selectExpr('employee_id as emp_id', 'name', 'cast(age as int) as age', 'salary')
emp_simple_df_1.printSchema()

root
 |-- emp_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- salary: string (nullable = true)



In [10]:
emp_simple_df_1.write.mode('overwrite').csv('/home/spark/data/employees_simple', header=True)

In [11]:
emp_df.write.mode('overwrite').csv('/home/spark/data/employees', header=True)